In [1]:
# ============================================================
# COMPOUND DROUGHT-HEATWAVE (CDHW) ANALYSIS
# South Africa, 1981–2025
#
# ============================================================

import os
import numpy as np
import xarray as xr


# ============================================================
# 1. INPUT FILES
# ============================================================

# Daily heatwave active mask
hw_daily_file = (
    r"C:\Users\franc\heatwave_daily_mask_1981_2025.nc"
)

# Daily flash-drought active mask
fd_daily_file = (
    r"C:\Users\franc\flash_drought_daily_mask_1981_2025.nc"
)

# Annual heatwave metrics
hw_metrics_file = (
    r"C:\Users\franc\south_africa_maize_season_heatwaves_1981_2025_updated.nc"
)

# Final annual flash-drought metrics
fd_metrics_file = (
    r"C:\Users\franc\flash_drought_events_1982_2025_final.nc"
)


# ============================================================
# 2. OUTPUT FILE
# ============================================================

output_file = (
    r"C:\Users\franc\south_africa_compound_extremes_summary_1982_2025.nc"
)


# ============================================================
# 3. CHECK FILES
# ============================================================

print("=" * 70)
print("COMPOUND DROUGHT-HEATWAVE ANALYSIS")
print("=" * 70)


files_to_check = {
    "Daily heatwave mask": hw_daily_file,
    "Daily flash drought mask": fd_daily_file,
    "Annual heatwave metrics": hw_metrics_file,
    "Annual flash drought metrics": fd_metrics_file
}


for name, path in files_to_check.items():

    print(
        f"\n{name}:"
    )

    print(path)

    print(
        "Exists:",
        os.path.exists(path)
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )


# ============================================================
# 4. OPEN DAILY MASKS
# ============================================================

print(
    "\nLoading daily heatwave and flash-drought masks..."
)


hw_ds = xr.open_dataset(
    hw_daily_file
)

fd_ds = xr.open_dataset(
    fd_daily_file
)


# ============================================================
# 5. SELECT VARIABLES
# ============================================================

if "heatwave_active" not in hw_ds.data_vars:

    raise ValueError(
        "Variable 'heatwave_active' "
        "not found in heatwave file."
    )


if "flash_drought_active" not in fd_ds.data_vars:

    raise ValueError(
        "Variable 'flash_drought_active' "
        "not found in drought file."
    )


hw_daily = hw_ds[
    "heatwave_active"
]


fd_daily = fd_ds[
    "flash_drought_active"
]


print("\nOriginal daily masks")

print(
    "Heatwave:",
    hw_daily.shape,
    hw_daily.dims
)

print(
    "Flash drought:",
    fd_daily.shape,
    fd_daily.dims
)


# ============================================================
# 6. STANDARDISE COORDINATE NAMES
# ============================================================

# Flash drought uses lat/lon.
# Heatwave uses latitude/longitude.

rename_dict = {}

if "lat" in fd_daily.dims:
    rename_dict["lat"] = "latitude"

if "lon" in fd_daily.dims:
    rename_dict["lon"] = "longitude"


if rename_dict:

    fd_daily = fd_daily.rename(
        rename_dict
    )


# ============================================================
# 7. CHECK TIME
# ============================================================

print("\nTime alignment check")
print("-" * 40)


print(
    "Heatwave:",
    hw_daily.time.values[0],
    "to",
    hw_daily.time.values[-1]
)


print(
    "Flash drought:",
    fd_daily.time.values[0],
    "to",
    fd_daily.time.values[-1]
)


print(
    "Heatwave days:",
    hw_daily.sizes["time"]
)


print(
    "Flash drought days:",
    fd_daily.sizes["time"]
)


times_identical = np.array_equal(
    hw_daily.time.values,
    fd_daily.time.values
)


print(
    "Time coordinates identical:",
    times_identical
)


if not times_identical:

    raise ValueError(
        "Heatwave and flash-drought "
        "time coordinates do not match."
    )


# ============================================================
# 8. SORT SPATIAL COORDINATES
# ============================================================

hw_daily = hw_daily.sortby(
    "latitude"
)

hw_daily = hw_daily.sortby(
    "longitude"
)


fd_daily = fd_daily.sortby(
    "latitude"
)

fd_daily = fd_daily.sortby(
    "longitude"
)


# ============================================================
# 9. CHECK SPATIAL GRIDS
# ============================================================

print("\nSpatial grids")
print("-" * 40)


print(
    "Heatwave grid:",
    hw_daily.sizes["latitude"],
    "x",
    hw_daily.sizes["longitude"]
)


print(
    "Flash drought grid:",
    fd_daily.sizes["latitude"],
    "x",
    fd_daily.sizes["longitude"]
)


print(
    "\nHeatwave latitude:",
    float(hw_daily.latitude.min()),
    "to",
    float(hw_daily.latitude.max())
)


print(
    "Heatwave longitude:",
    float(hw_daily.longitude.min()),
    "to",
    float(hw_daily.longitude.max())
)


print(
    "\nFlash drought latitude:",
    float(fd_daily.latitude.min()),
    "to",
    float(fd_daily.latitude.max())
)


print(
    "Flash drought longitude:",
    float(fd_daily.longitude.min()),
    "to",
    float(fd_daily.longitude.max())
)


# ============================================================
# 10. REGRID FLASH DROUGHT TO HEATWAVE GRID
# ============================================================
#
# Heatwave grid = 57 × 89
# Flash drought = 141 × 220
#
# We use the heatwave grid as the common grid.
#
# Since drought active is categorical 0/1,
# nearest-neighbour interpolation is appropriate.
# ============================================================

print(
    "\nRegridding flash-drought mask "
    "onto heatwave grid..."
)


fd_daily_regridded = fd_daily.interp(

    latitude=
        hw_daily.latitude,

    longitude=
        hw_daily.longitude,

    method="nearest"
)


fd_daily_regridded = (
    fd_daily_regridded
    .fillna(0)
    .astype(np.int8)
)


hw_daily = hw_daily.astype(
    np.int8
)


print(
    "Regridded flash drought:",
    fd_daily_regridded.shape
)


print(
    "Heatwave:",
    hw_daily.shape
)


if (
    fd_daily_regridded.shape
    !=
    hw_daily.shape
):

    raise ValueError(
        "Daily masks still have different shapes."
    )


# ============================================================
# 11. SIMULTANEOUS CDHW
# ============================================================
#
# Both flash drought and heatwave are active
# on the SAME pixel and SAME day.
# ============================================================

print(
    "\nCalculating simultaneous CDHW days..."
)


cdhw_simultaneous = xr.where(

    (hw_daily == 1)
    &
    (fd_daily_regridded == 1),

    1,

    0

).astype(
    np.int8
)


cdhw_simultaneous.name = (
    "cdhw_simultaneous_active"
)


# ============================================================
# 12. SEQUENTIAL CDHW
# ============================================================
#
# Supervisor's definition:
#
# flash drought active today
# AND
# heatwave occurred within previous 7 days
# AND
# today is not itself a heatwave day.
# ============================================================

print(
    "Calculating sequential CDHW events..."
)


hw_rolling_7d = (
    hw_daily
    .rolling(
        time=7,
        min_periods=1
    )
    .sum()
)


cdhw_sequential = xr.where(

    (fd_daily_regridded == 1)
    &
    (hw_rolling_7d > 0)
    &
    (hw_daily == 0),

    1,

    0

).astype(
    np.int8
)


cdhw_sequential.name = (
    "cdhw_sequential_active"
)


# ============================================================
# 13. TOTAL COMPOUND STATE
# ============================================================

compound_state = xr.where(

    (cdhw_simultaneous == 1)
    |
    (cdhw_sequential == 1),

    1,

    0

).astype(
    np.int8
)


compound_state.name = (
    "cdhw_active"
)


# ============================================================
# 14. HARVEST-YEAR COORDINATE
# ============================================================

harvest_year = (

    compound_state.time.dt.year

    +

    xr.where(
        compound_state.time.dt.month >= 10,
        1,
        0
    )
)


cdhw_simultaneous = (
    cdhw_simultaneous
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


cdhw_sequential = (
    cdhw_sequential
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


compound_state = (
    compound_state
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


# ============================================================
# 15. KEEP COMPLETE HARVEST YEARS
# ============================================================

complete_mask = (

    (compound_state.harvest_year >= 1982)

    &

    (compound_state.harvest_year <= 2025)
)


cdhw_simultaneous_complete = (
    cdhw_simultaneous.sel(
        time=complete_mask
    )
)


cdhw_sequential_complete = (
    cdhw_sequential.sel(
        time=complete_mask
    )
)


compound_complete = (
    compound_state.sel(
        time=complete_mask
    )
)


# ============================================================
# 16. ANNUAL SIMULTANEOUS DAYS
# ============================================================

print(
    "\nAggregating compound metrics "
    "by harvest year..."
)


annual_simultaneous_days = (

    cdhw_simultaneous_complete

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )

)


annual_simultaneous_days.name = (
    "cdhw_simultaneous_days"
)


# ============================================================
# 17. ANNUAL SEQUENTIAL DAYS
# ============================================================

annual_sequential_days = (

    cdhw_sequential_complete

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )

)


annual_sequential_days.name = (
    "cdhw_sequential_days"
)


# ============================================================
# 18. COMPOUND EVENT FREQUENCY
# ============================================================
#
# Event start:
# current compound day = 1
# previous day = 0
#
# This is slightly safer than diff()==1.
# ============================================================

previous_compound = (
    compound_complete.shift(
        time=1,
        fill_value=0
    )
)


compound_starts = (

    (compound_complete == 1)

    &

    (previous_compound == 0)

).astype(
    np.int8
)


annual_compound_events = (

    compound_starts

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )

)


annual_compound_events.name = (
    "cdhw_total_events"
)


# ============================================================
# 19. TOTAL COMPOUND DAYS
# ============================================================

annual_compound_days = (

    compound_complete

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )

)


annual_compound_days.name = (
    "cdhw_total_days"
)


# ============================================================
# 20. OPEN ANNUAL HEATWAVE METRICS
# ============================================================

print(
    "\nLoading annual hazard metrics "
    "for joint-risk index..."
)


hw_metrics_ds = xr.open_dataset(
    hw_metrics_file
)


fd_metrics_ds = xr.open_dataset(
    fd_metrics_file,
    decode_timedelta=False
)


hw_frequency = (
    hw_metrics_ds[
        "hw_frequency"
    ]
)


fd_frequency = (
    fd_metrics_ds[
        "fd_frequency"
    ]
)


# ============================================================
# 21. STANDARDISE FD COORDINATE NAMES
# ============================================================

fd_metric_rename = {}

if "lat" in fd_frequency.dims:

    fd_metric_rename[
        "lat"
    ] = "latitude"


if "lon" in fd_frequency.dims:

    fd_metric_rename[
        "lon"
    ] = "longitude"


if fd_metric_rename:

    fd_frequency = fd_frequency.rename(
        fd_metric_rename
    )


# ============================================================
# 22. SORT ANNUAL GRIDS
# ============================================================

hw_frequency = (
    hw_frequency
    .sortby("latitude")
    .sortby("longitude")
)


fd_frequency = (
    fd_frequency
    .sortby("latitude")
    .sortby("longitude")
)


# ============================================================
# 23. REGRID FD FREQUENCY TO HEATWAVE GRID
# ============================================================

fd_frequency_regridded = (
    fd_frequency.interp(

        latitude=
            hw_frequency.latitude,

        longitude=
            hw_frequency.longitude,

        method="nearest"
    )
)


# ============================================================
# 24. ALIGN HARVEST YEARS
# ============================================================

hw_frequency = hw_frequency.sel(
    harvest_year=slice(
        1982,
        2025
    )
)


fd_frequency_regridded = (
    fd_frequency_regridded.sel(
        harvest_year=slice(
            1982,
            2025
        )
    )
)


hw_frequency, fd_frequency_regridded = (
    xr.align(
        hw_frequency,
        fd_frequency_regridded,
        join="exact"
    )
)


# ============================================================
# 25. JOINT RISK INDEX
# ============================================================
#
# Follows supervisor's code:
#
# heatwave frequency × flash drought frequency
# ============================================================

joint_risk_index = (

    hw_frequency

    *

    fd_frequency_regridded

)


joint_risk_index.name = (
    "joint_risk_index"
)


# ============================================================
# 26. BUILD FINAL COMPOUND DATASET
# ============================================================

compound_ds = xr.Dataset(
    {

        "cdhw_simultaneous_days":
            annual_simultaneous_days,

        "cdhw_sequential_days":
            annual_sequential_days,

        "cdhw_total_days":
            annual_compound_days,

        "cdhw_total_events":
            annual_compound_events,

        "joint_risk_index":
            joint_risk_index

    }
)


# ============================================================
# 27. ATTRIBUTES
# ============================================================

compound_ds[
    "cdhw_simultaneous_days"
].attrs["long_name"] = (
    "Number of days with simultaneous "
    "heatwave and flash drought"
)


compound_ds[
    "cdhw_sequential_days"
].attrs["long_name"] = (
    "Number of flash-drought days preceded "
    "by a heatwave within 7 days"
)


compound_ds[
    "cdhw_total_days"
].attrs["long_name"] = (
    "Total compound drought-heatwave days"
)


compound_ds[
    "cdhw_total_events"
].attrs["long_name"] = (
    "Number of distinct compound "
    "drought-heatwave events"
)


compound_ds[
    "joint_risk_index"
].attrs["long_name"] = (
    "Heatwave frequency multiplied by "
    "flash-drought frequency"
)


# ============================================================
# 28. FINAL CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL CDHW METRICS"
)

print(
    "=" * 70
)


print(
    "Dimensions:"
)

print(
    dict(
        compound_ds.sizes
    )
)


print(
    "\nFirst harvest year:",
    int(
        compound_ds
        .harvest_year
        .values[0]
    )
)


print(
    "Last harvest year:",
    int(
        compound_ds
        .harvest_year
        .values[-1]
    )
)


print(
    "\nMaximum simultaneous days:",
    int(
        compound_ds[
            "cdhw_simultaneous_days"
        ].max().values
    )
)


print(
    "Maximum sequential days:",
    int(
        compound_ds[
            "cdhw_sequential_days"
        ].max().values
    )
)


print(
    "Maximum total compound days:",
    int(
        compound_ds[
            "cdhw_total_days"
        ].max().values
    )
)


print(
    "Maximum compound events:",
    int(
        compound_ds[
            "cdhw_total_events"
        ].max().values
    )
)


print(
    "Maximum joint risk index:",
    float(
        compound_ds[
            "joint_risk_index"
        ].max().values
    )
)


# ============================================================
# 29. TEST LOCATION
# ============================================================

test = compound_ds.sel(

    latitude=-27.95,

    longitude=21.05,

    method="nearest"
)


print(
    "\nTest location"
)

print(
    "-" * 40
)


print(
    "Latitude:",
    float(test.latitude)
)


print(
    "Longitude:",
    float(test.longitude)
)


print(
    "\nFirst 10 harvest years:"
)

print(
    test.harvest_year.values[:10]
)


print(
    "\nSimultaneous days:"
)

print(
    test[
        "cdhw_simultaneous_days"
    ].values[:10]
)


print(
    "\nSequential days:"
)

print(
    test[
        "cdhw_sequential_days"
    ].values[:10]
)


print(
    "\nTotal compound days:"
)

print(
    test[
        "cdhw_total_days"
    ].values[:10]
)


print(
    "\nCompound events:"
)

print(
    test[
        "cdhw_total_events"
    ].values[:10]
)


# ============================================================
# 30. SAVE FINAL COMPOUND DATASET
# ============================================================

print(
    "\nSaving compound dataset..."
)


compound_ds.to_netcdf(
    output_file
)


print(
    "\nSaved successfully:"
)

print(
    output_file
)


print(
    "\nDONE!"
)

COMPOUND DROUGHT-HEATWAVE ANALYSIS

Daily heatwave mask:
C:\Users\franc\heatwave_daily_mask_1981_2025.nc
Exists: True

Daily flash drought mask:
C:\Users\franc\flash_drought_daily_mask_1981_2025.nc
Exists: True

Annual heatwave metrics:
C:\Users\franc\south_africa_maize_season_heatwaves_1981_2025_updated.nc
Exists: True

Annual flash drought metrics:
C:\Users\franc\flash_drought_events_1982_2025_final.nc
Exists: True

Loading daily heatwave and flash-drought masks...

Original daily masks
Heatwave: (16436, 57, 89) ('time', 'latitude', 'longitude')
Flash drought: (16436, 141, 220) ('time', 'lat', 'lon')

Time alignment check
----------------------------------------
Heatwave: 1981-01-01T00:00:00.000000000 to 2025-12-31T00:00:00.000000000
Flash drought: 1981-01-01T00:00:00.000000000 to 2025-12-31T00:00:00.000000000
Heatwave days: 16436
Flash drought days: 16436
Time coordinates identical: True

Spatial grids
----------------------------------------
Heatwave grid: 57 x 89
Flash drought gri

In [4]:
# ============================================================
# CDHW SEVERITY / INTENSITY ANALYSIS
# South Africa maize seasons, 1982–2025
# ============================================================

import os
import numpy as np
import xarray as xr


# ============================================================
# 1. FILE PATHS
# ============================================================

temperature_file = (
    r"C:\Users\franc\era5_daily_max_2m_temperature.nc"
)

spei_file = (
    r"C:\Users\franc\spei1_spei3_1981_2025.nc"
)

hw_mask_file = (
    r"C:\Users\franc\heatwave_daily_mask_1981_2025.nc"
)

fd_mask_file = (
    r"C:\Users\franc\flash_drought_daily_mask_1981_2025.nc"
)

compound_file = (
    r"C:\Users\franc\south_africa_compound_extremes_summary_1982_2025.nc"
)

output_file = (
    r"C:\Users\franc\cdhw_severity_1982_2025.nc"
)


# ============================================================
# 2. CHECK FILES
# ============================================================

print("=" * 70)
print("CDHW SEVERITY ANALYSIS")
print("=" * 70)

files = {
    "Temperature": temperature_file,
    "SPEI": spei_file,
    "Heatwave mask": hw_mask_file,
    "Flash-drought mask": fd_mask_file,
    "Compound summary": compound_file
}

for name, path in files.items():

    print(f"\n{name}:")
    print(path)
    print("Exists:", os.path.exists(path))

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} file not found:\n{path}"
        )


# ============================================================
# 3. OPEN DATASETS
# ============================================================

print("\nOpening datasets...")

temp_ds = xr.open_dataset(
    temperature_file
)

spei_ds = xr.open_dataset(
    spei_file
)

hw_ds = xr.open_dataset(
    hw_mask_file
)

fd_ds = xr.open_dataset(
    fd_mask_file
)


# ============================================================
# 4. SELECT VARIABLES
# ============================================================

tmax = temp_ds["t2m"]

spei3 = spei_ds["spei3"]

hw = hw_ds["heatwave_active"]

fd = fd_ds["flash_drought_active"]


print("\nOriginal shapes")
print("-" * 40)

print(
    "Temperature:",
    tmax.shape
)

print(
    "SPEI-3:",
    spei3.shape
)

print(
    "Heatwave:",
    hw.shape
)

print(
    "Flash drought:",
    fd.shape
)


# ============================================================
# 5. CONVERT TEMPERATURE TO CELSIUS
# ============================================================

if (
    tmax.attrs.get("units") == "K"
    or float(tmax.max(skipna=True).values) > 200
):

    print("\nConverting temperature to Celsius...")

    tmax = tmax - 273.15

    tmax.attrs["units"] = "degrees_Celsius"


# ============================================================
# 6. STANDARDISE DROUGHT COORDINATE NAMES
# ============================================================

rename_spei = {}

if "lat" in spei3.dims:
    rename_spei["lat"] = "latitude"

if "lon" in spei3.dims:
    rename_spei["lon"] = "longitude"

if rename_spei:
    spei3 = spei3.rename(rename_spei)


rename_fd = {}

if "lat" in fd.dims:
    rename_fd["lat"] = "latitude"

if "lon" in fd.dims:
    rename_fd["lon"] = "longitude"

if rename_fd:
    fd = fd.rename(rename_fd)


# ============================================================
# 7. SORT GRIDS
# ============================================================

tmax = (
    tmax
    .sortby("latitude")
    .sortby("longitude")
)

hw = (
    hw
    .sortby("latitude")
    .sortby("longitude")
)

spei3 = (
    spei3
    .sortby("latitude")
    .sortby("longitude")
)

fd = (
    fd
    .sortby("latitude")
    .sortby("longitude")
)


# ============================================================
# 8. CHECK TIME ALIGNMENT
# ============================================================

print("\nTime alignment")
print("-" * 40)

print(
    "Temperature days:",
    tmax.sizes["time"]
)

print(
    "SPEI days:",
    spei3.sizes["time"]
)

print(
    "Heatwave days:",
    hw.sizes["time"]
)

print(
    "Drought days:",
    fd.sizes["time"]
)


if not np.array_equal(
    tmax.time.values,
    hw.time.values
):
    raise ValueError(
        "Temperature and heatwave dates do not match."
    )


if not np.array_equal(
    spei3.time.values,
    fd.time.values
):
    raise ValueError(
        "SPEI and drought-mask dates do not match."
    )


if not np.array_equal(
    tmax.time.values,
    spei3.time.values
):
    raise ValueError(
        "Temperature and SPEI dates do not match."
    )


print("Time coordinates match!")


# ============================================================
# 9. REGRID DROUGHT VARIABLES TO ERA5 TEMPERATURE GRID
# ============================================================
#
# Temperature / heatwave:
# 57 x 89
#
# SPEI / drought:
# 141 x 220
#
# SPEI is continuous -> linear interpolation
# Drought mask is categorical -> nearest neighbour
# ============================================================

print(
    "\nRegridding SPEI-3 to heatwave grid..."
)

spei3_hw = spei3.interp(

    latitude=tmax.latitude,

    longitude=tmax.longitude,

    method="linear"
)


print(
    "Regridding drought mask to heatwave grid..."
)

fd_hw = fd.interp(

    latitude=tmax.latitude,

    longitude=tmax.longitude,

    method="nearest"
)


fd_hw = (
    fd_hw
    .fillna(0)
    .astype(np.int8)
)


print("\nAligned shapes")
print("-" * 40)

print(
    "Temperature:",
    tmax.shape
)

print(
    "Heatwave:",
    hw.shape
)

print(
    "SPEI-3:",
    spei3_hw.shape
)

print(
    "Drought:",
    fd_hw.shape
)


# ============================================================
# 10. CALCULATE HEATWAVE THRESHOLD
# ============================================================

print(
    "\nCalculating 90th-percentile "
    "temperature threshold..."
)

growing_months = [
    10, 11, 12,
    1, 2, 3, 4
]


growing = (
    tmax.time.dt.month.isin(
        growing_months
    )
)


tmax_growing = tmax.sel(
    time=growing
)


threshold = (

    tmax_growing

    .groupby(
        "time.dayofyear"
    )

    .quantile(
        0.90,
        dim="time"
    )
)


# ============================================================
# 11. TEMPERATURE EXCESS
# ============================================================
#
# Threshold climatology exists only for Oct-Apr.
# Therefore calculate excess only on growing-season dates,
# then place it back onto the full daily calendar.
# ============================================================

print(
    "Calculating heatwave temperature excess..."
)

# ------------------------------------------------------------
# Growing-season temperature
# ------------------------------------------------------------

tmax_growing_full = tmax.sel(
    time=growing
)

hw_growing = hw.sel(
    time=growing
)


# ------------------------------------------------------------
# Match each growing-season date to its daily threshold
# ------------------------------------------------------------

threshold_growing_daily = threshold.sel(
    dayofyear=
        tmax_growing_full.time.dt.dayofyear
)


# ------------------------------------------------------------
# Temperature excess on growing-season days
# ------------------------------------------------------------

heat_excess_growing = (
    tmax_growing_full
    -
    threshold_growing_daily
)


# Keep excess only during actual heatwave days
heat_excess_growing = xr.where(
    hw_growing == 1,
    heat_excess_growing,
    0
)


# Remove any negative values
heat_excess_growing = xr.where(
    heat_excess_growing > 0,
    heat_excess_growing,
    0
)


# ------------------------------------------------------------
# Create full-calendar heat-excess array
# May-Sep remain zero
# ------------------------------------------------------------

heat_excess = xr.zeros_like(
    tmax,
    dtype=np.float32
)


heat_excess.loc[
    {
        "time":
            tmax_growing_full.time
    }
] = heat_excess_growing.astype(
    np.float32
)


print(
    "Heatwave temperature excess calculated successfully."
)

print(
    "Heat excess shape:",
    heat_excess.shape
)

print(
    "Maximum heat excess:",
    float(
        heat_excess.max().values
    )
) 


# ============================================================
# 12. DROUGHT DEFICIT
# ============================================================
#
# SPEI is negative during drought.
#
# Convert negative drought values into
# positive deficit magnitude.
#
# Example:
#
# SPEI-3 = -1.8
# drought deficit = 1.8
# ============================================================

print(
    "Calculating drought deficit..."
)


drought_deficit = xr.where(

    fd_hw == 1,

    np.maximum(
        -spei3_hw,
        0
    ),

    0
)


# ============================================================
# 13. SIMULTANEOUS CDHW MASK
# ============================================================

cdhw = xr.where(

    (hw == 1)
    &
    (fd_hw == 1),

    1,

    0

).astype(np.int8)


# ============================================================
# 14. DAILY CDHW COMPONENT INTENSITIES
# ============================================================

cdhw_heat_intensity = xr.where(

    cdhw == 1,

    heat_excess,

    0
)


cdhw_drought_intensity = xr.where(

    cdhw == 1,

    drought_deficit,

    0
)


# ============================================================
# 15. DAILY COMBINED SEVERITY
# ============================================================
#
# Multiplicative joint severity:
#
# temperature excess × SPEI deficit
#
# Therefore a day is severe only when BOTH
# components are strong.
# ============================================================

cdhw_daily_severity = (

    cdhw_heat_intensity

    *

    cdhw_drought_intensity
)


cdhw_daily_severity.name = (
    "cdhw_daily_severity"
)


# ============================================================
# 16. ASSIGN HARVEST YEAR
# ============================================================

harvest_year = (

    tmax.time.dt.year

    +

    xr.where(
        tmax.time.dt.month >= 10,
        1,
        0
    )
)


cdhw = cdhw.assign_coords(

    harvest_year=(
        "time",
        harvest_year.values
    )
)


cdhw_heat_intensity = (
    cdhw_heat_intensity
    .assign_coords(

        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


cdhw_drought_intensity = (
    cdhw_drought_intensity
    .assign_coords(

        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


cdhw_daily_severity = (
    cdhw_daily_severity
    .assign_coords(

        harvest_year=(
            "time",
            harvest_year.values
        )
    )
)


# ============================================================
# 17. KEEP MAIZE SEASON ONLY
# ============================================================

season_mask = (

    tmax.time.dt.month.isin(
        growing_months
    )

    &

    (harvest_year >= 1982)

    &

    (harvest_year <= 2025)
)


cdhw_season = cdhw.sel(
    time=season_mask
)


heat_season = (
    cdhw_heat_intensity
    .sel(
        time=season_mask
    )
)


drought_season = (
    cdhw_drought_intensity
    .sel(
        time=season_mask
    )
)


severity_season = (
    cdhw_daily_severity
    .sel(
        time=season_mask
    )
)


# ============================================================
# 18. ANNUAL CDHW DAYS
# ============================================================

print(
    "\nAggregating severity by harvest year..."
)


annual_cdhw_days = (

    cdhw_season

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )
)


# ============================================================
# 19. CUMULATIVE HEAT INTENSITY
# ============================================================

annual_heat_intensity = (

    heat_season

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )
)


# ============================================================
# 20. CUMULATIVE DROUGHT INTENSITY
# ============================================================

annual_drought_intensity = (

    drought_season

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )
)


# ============================================================
# 21. CUMULATIVE JOINT SEVERITY
# ============================================================

annual_joint_severity = (

    severity_season

    .groupby(
        "harvest_year"
    )

    .sum(
        "time"
    )
)


# ============================================================
# 22. PEAK DAILY JOINT SEVERITY
# ============================================================

peak_joint_severity = (

    severity_season

    .groupby(
        "harvest_year"
    )

    .max(
        "time"
    )
)


# ============================================================
# 23. MEAN SEVERITY PER CDHW DAY
# ============================================================

mean_joint_severity = xr.where(

    annual_cdhw_days > 0,

    annual_joint_severity
    /
    annual_cdhw_days,

    0
)


# ============================================================
# 24. CREATE DATASET
# ============================================================

severity_ds = xr.Dataset(
    {

        "cdhw_simultaneous_days":
            annual_cdhw_days,

        "cdhw_heat_intensity":
            annual_heat_intensity,

        "cdhw_drought_intensity":
            annual_drought_intensity,

        "cdhw_cumulative_severity":
            annual_joint_severity,

        "cdhw_peak_severity":
            peak_joint_severity,

        "cdhw_mean_severity":
            mean_joint_severity

    }
)


# ============================================================
# 25. ATTRIBUTES
# ============================================================

severity_ds[
    "cdhw_heat_intensity"
].attrs["description"] = (
    "Cumulative Tmax excess above the "
    "daily 90th-percentile threshold during "
    "simultaneous CDHW days"
)


severity_ds[
    "cdhw_drought_intensity"
].attrs["description"] = (
    "Cumulative positive magnitude of negative "
    "SPEI-3 during simultaneous CDHW days"
)


severity_ds[
    "cdhw_cumulative_severity"
].attrs["description"] = (
    "Cumulative product of heatwave temperature "
    "excess and SPEI-3 drought deficit during "
    "simultaneous compound days"
)


severity_ds[
    "cdhw_peak_severity"
].attrs["description"] = (
    "Maximum daily joint CDHW severity "
    "during each harvest year"
)


severity_ds[
    "cdhw_mean_severity"
].attrs["description"] = (
    "Mean joint severity per simultaneous "
    "CDHW day"
)


# ============================================================
# 26. FINAL CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL CDHW SEVERITY METRICS"
)

print(
    "=" * 70
)


print(
    "Dimensions:",
    dict(
        severity_ds.sizes
    )
)


print(
    "\nFirst harvest year:",
    int(
        severity_ds.harvest_year.values[0]
    )
)


print(
    "Last harvest year:",
    int(
        severity_ds.harvest_year.values[-1]
    )
)


print(
    "\nMaximum simultaneous days:",
    int(
        severity_ds[
            "cdhw_simultaneous_days"
        ].max().values
    )
)


print(
    "Maximum heat intensity:",
    float(
        severity_ds[
            "cdhw_heat_intensity"
        ].max().values
    )
)


print(
    "Maximum drought intensity:",
    float(
        severity_ds[
            "cdhw_drought_intensity"
        ].max().values
    )
)


print(
    "Maximum cumulative severity:",
    float(
        severity_ds[
            "cdhw_cumulative_severity"
        ].max().values
    )
)


print(
    "Maximum peak severity:",
    float(
        severity_ds[
            "cdhw_peak_severity"
        ].max().values
    )
)


print(
    "Maximum mean severity:",
    float(
        severity_ds[
            "cdhw_mean_severity"
        ].max().values
    )
)


# ============================================================
# 27. TEST LOCATION
# ============================================================

test = severity_ds.sel(

    latitude=-27.95,

    longitude=21.05,

    method="nearest"
)


print(
    "\nTest location"
)

print(
    "-" * 40
)


print(
    "Latitude:",
    float(test.latitude)
)


print(
    "Longitude:",
    float(test.longitude)
)


print(
    "\nFirst 10 harvest years:"
)

print(
    test.harvest_year.values[:10]
)


print(
    "\nCDHW days:"
)

print(
    test[
        "cdhw_simultaneous_days"
    ].values[:10]
)


print(
    "\nCumulative severity:"
)

print(
    test[
        "cdhw_cumulative_severity"
    ].values[:10]
)


# ============================================================
# 28. SAVE
# ============================================================

print(
    "\nSaving CDHW severity dataset..."
)


severity_ds.to_netcdf(
    output_file
)


print(
    "\nSaved successfully:"
)

print(
    output_file
)


print(
    "\nDONE!"
)

CDHW SEVERITY ANALYSIS

Temperature:
C:\Users\franc\era5_daily_max_2m_temperature.nc
Exists: True

SPEI:
C:\Users\franc\spei1_spei3_1981_2025.nc
Exists: True

Heatwave mask:
C:\Users\franc\heatwave_daily_mask_1981_2025.nc
Exists: True

Flash-drought mask:
C:\Users\franc\flash_drought_daily_mask_1981_2025.nc
Exists: True

Compound summary:
C:\Users\franc\south_africa_compound_extremes_summary_1982_2025.nc
Exists: True

Opening datasets...

Original shapes
----------------------------------------
Temperature: (16436, 57, 89)
SPEI-3: (16436, 141, 220)
Heatwave: (16436, 57, 89)
Flash drought: (16436, 141, 220)

Converting temperature to Celsius...

Time alignment
----------------------------------------
Temperature days: 16436
SPEI days: 16436
Heatwave days: 16436
Drought days: 16436
Time coordinates match!

Regridding SPEI-3 to heatwave grid...
Regridding drought mask to heatwave grid...

Aligned shapes
----------------------------------------
Temperature: (16436, 57, 89)
Heatwave: (16436